# MS Data Correction

This notebook cleans the EDSS (Expanded Disability Status Scale, code `273554001`) observations in the raw Synthea MS dataset at `data/raw/csv` and writes a corrected copy to `data/corrected`. **The raw files are never modified.**

Pipeline:
1. Count the patients that have a negative EDSS value.
2. Remove those patients entirely, from every CSV file.
3. On what remains, check that every EDSS value lies in `[0, 10]` and lands on a `0.5` step, and count how many patients still violate this.
4. Correct the values that are off the `0.5` grid (round, don't drop).
5. Save the corrected dataset.

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw/csv")
CORRECTED_DIR = Path("../data/corrected")
CORRECTED_DIR.mkdir(parents=True, exist_ok=True)

EDSS_CODE = "273554001"

# Column that identifies the patient in each CSV file; None means the file has no per-patient rows
PATIENT_ID_COLUMNS = {
    "claims.csv": "PATIENTID",
    "claims_transactions.csv": "PATIENTID",
    "conditions.csv": "PATIENT",
    "encounters.csv": "PATIENT",
    "immunizations.csv": "PATIENT",
    "medications.csv": "PATIENT",
    "observations.csv": "PATIENT",
    "patients.csv": "Id",
    "payer_transitions.csv": "PATIENT",
    "procedures.csv": "PATIENT",
    "organizations.csv": None,
    "payers.csv": None,
    "providers.csv": None,
}

# Load every raw file once; we mutate this dict in place as the pipeline progresses
dataframes = {csv_path.name: pd.read_csv(csv_path) for csv_path in sorted(RAW_DIR.glob("*.csv"))}

missing = set(dataframes) - set(PATIENT_ID_COLUMNS)
assert not missing, f"Unmapped CSV files found: {missing}"

## 1. Patients with a negative EDSS value

EDSS is clinically defined on `[0, 10]`, so a negative score is not a real measurement — it's a sign that the synthetic generator produced a bad value for that patient. We isolate the EDSS rows (`CODE == "273554001"`), coerce `VALUE` to numeric, and collect the set of `PATIENT` ids that have **at least one** negative reading.

In [ ]:
observations = dataframes["observations.csv"]
edss = observations[observations["CODE"].astype(str) == EDSS_CODE].copy()
edss["VALUE"] = pd.to_numeric(edss["VALUE"], errors="coerce")

patients_with_negative_edss = set(edss.loc[edss["VALUE"] < 0, "PATIENT"].unique())

print(f"EDSS observations (raw):              {len(edss)}")
print(f"Negative EDSS observations:            {(edss['VALUE'] < 0).sum()}")
print(f"Patients with a negative EDSS value:   {len(patients_with_negative_edss)}")
patients_with_negative_edss

## 2. Remove these patients

A negative EDSS reading calls the patient's *whole* disease-trajectory record into question, not just that one observation. So we drop **every row for these patients, in every file** — claims, encounters, medications, the patient record itself, etc. — not only the offending observation.

Each file is filtered on its own patient-id column from `PATIENT_ID_COLUMNS`. Files with no patient column (`organizations.csv`, `payers.csv`, `providers.csv`) are reference tables shared across patients, so they're left untouched.

In [ ]:
rows_removed = {}
for name, patient_col in PATIENT_ID_COLUMNS.items():
    if patient_col is None:
        rows_removed[name] = 0
        continue
    df = dataframes[name]
    mask = df[patient_col].isin(patients_with_negative_edss)
    rows_removed[name] = int(mask.sum())
    dataframes[name] = df.loc[~mask].reset_index(drop=True)

# Refresh the EDSS view now that those patients are gone
observations = dataframes["observations.csv"]
edss = observations[observations["CODE"].astype(str) == EDSS_CODE].copy()
edss["VALUE"] = pd.to_numeric(edss["VALUE"], errors="coerce")

print(f"Patients removed: {len(patients_with_negative_edss)}")
print(f"Remaining EDSS observations: {len(edss)}")
pd.Series(rows_removed, name="rows_removed").sort_index()

## 3. Validate the remaining EDSS values

Two independent checks on what's left, now that the bad patients are gone:

- **Range** — every value must lie within `[0, 10]` inclusive.
- **Step** — EDSS is only ever recorded in `0.5` increments (`0.0, 0.5, 1.0, …, 10.0`); a value like `2.7` is not a grade a clinician would actually record.

We report both the *observation*-level counts and the number of *distinct patients* affected, since one non-compliant patient can contribute several bad observations.

In [ ]:
out_of_range = edss[(edss["VALUE"] < 0) | (edss["VALUE"] > 10) | edss["VALUE"].isna()]

steps = edss["VALUE"] / 0.5
off_step = edss[~steps.round(6).eq(steps.round()) | edss["VALUE"].isna()]

non_compliant = pd.concat([out_of_range, off_step]).drop_duplicates()
non_compliant_patients = set(non_compliant["PATIENT"].unique())

print(f"EDSS observations checked:                    {len(edss)}")
print(f"Out of [0, 10] range:                          {len(out_of_range)}")
print(f"Not on a 0.5 step:                              {len(off_step)}")
print(f"Patients with a non-compliant EDSS value:      {len(non_compliant_patients)}")
non_compliant.sort_values("PATIENT").head(10)

## 4. Correct values that are not on a 0.5 step

**Approach:** round each EDSS value to the nearest `0.5` with `round(value * 2) / 2`.

- Multiplying by `2` maps the `0.5` grid onto the integers (`0, 1, 2, … `→ EDSS `0.0, 0.5, 1.0, …`).
- `round()` snaps to the nearest integer. Pandas/NumPy use *round-half-to-even* ("banker's rounding") for exact `.5` ties, e.g. a doubled value of `4.5` rounds to `4`, not `5` — this avoids a systematic upward bias on ties instead of always rounding up.
- Dividing by `2` maps back onto the `0.5` grid.

Worked examples: `2.7 → 5.4 → 5 → 2.5`, `2.3 → 4.6 → 5 → 2.5`, and the tie case `2.25 → 4.5 → 4 → 2.0`.

We then `clip` the result to `[0, 10]` as a safety net for the range check — this dataset happens to have no values above `10` after step 2, but clipping keeps the step correct if it's ever re-run on different data. Only the `VALUE` column changes: the observation itself is **kept**, since the underlying measurement is still informative once snapped to a valid grade — this is a correction, not a removal, which is why it's handled differently from step 1's negative-EDSS patients.

In [ ]:
is_edss = observations["CODE"].astype(str) == EDSS_CODE
raw_values = pd.to_numeric(observations.loc[is_edss, "VALUE"], errors="coerce")

corrected_values = ((raw_values * 2).round() / 2).clip(lower=0, upper=10)
changed = corrected_values.ne(raw_values)

print(f"EDSS values corrected: {int(changed.sum())} of {int(is_edss.sum())}")

observations.loc[is_edss, "VALUE"] = corrected_values
dataframes["observations.csv"] = observations

# Re-validate: both checks should now report zero violations
edss_after = observations.loc[is_edss].copy()
edss_after["VALUE"] = pd.to_numeric(edss_after["VALUE"], errors="coerce")
steps_after = edss_after["VALUE"] / 0.5

still_out_of_range = ((edss_after["VALUE"] < 0) | (edss_after["VALUE"] > 10)).sum()
still_off_step = (~steps_after.round(6).eq(steps_after.round())).sum()

print(f"Remaining out-of-range values: {still_out_of_range}")
print(f"Remaining off-step values:     {still_off_step}")

## 5. Save the corrected dataset

Write every dataframe in `dataframes` — patients with a negative EDSS removed (step 2), remaining EDSS values snapped to a valid `0.5` step (step 4) — to `data/corrected/`, one CSV per file, same filenames as the raw data. `data/raw/csv` itself is left untouched throughout this notebook.

In [ ]:
summary = []
for name, df in dataframes.items():
    df.to_csv(CORRECTED_DIR / name, index=False)
    summary.append({"file": name, "rows_final": len(df)})

pd.DataFrame(summary).sort_values("file").reset_index(drop=True)

## 6. Data test: validate `data/corrected` on disk

Independent check that re-reads `observations.csv` from `data/corrected` (not the in-memory `dataframes`) and asserts, for every EDSS observation, that the value is:

1. not negative,
2. within `[0, 10]`,
3. on a `0.5` step.

This is a regression test for the pipeline: if a future edit to steps 1–5 reintroduces a bad value, this cell raises `AssertionError` instead of silently writing the file.

In [ ]:
corrected_observations = pd.read_csv(CORRECTED_DIR / "observations.csv")
corrected_edss = corrected_observations[corrected_observations["CODE"].astype(str) == EDSS_CODE].copy()
corrected_edss["VALUE"] = pd.to_numeric(corrected_edss["VALUE"], errors="coerce")

is_negative = corrected_edss["VALUE"] < 0
is_in_range = corrected_edss["VALUE"].between(0, 10)
steps = corrected_edss["VALUE"] / 0.5
is_on_step = steps.round(6).eq(steps.round())

print(f"EDSS observations in data/corrected: {len(corrected_edss)}")
print(f"Negative values:      {int(is_negative.sum())}")
print(f"Outside [0, 10]:      {int((~is_in_range).sum())}")
print(f"Not on a 0.5 step:    {int((~is_on_step).sum())}")

assert corrected_edss["VALUE"].notna().all(), "Non-numeric EDSS value found in data/corrected"
assert not is_negative.any(), "Negative EDSS value found in data/corrected"
assert is_in_range.all(), "EDSS value outside [0, 10] found in data/corrected"
assert is_on_step.all(), "EDSS value not on a 0.5 step found in data/corrected"

print("\nAll data/corrected EDSS values are non-negative, within [0, 10], and on a 0.5 step.")

In [ ]:
raw_patient_count = pd.read_csv(RAW_DIR / "patients.csv").shape[0]
corrected_patient_count = pd.read_csv(CORRECTED_DIR / "patients.csv").shape[0]

print(f"Patients in data/raw/csv:         {raw_patient_count}")
print(f"Patients removed (negative EDSS): {len(patients_with_negative_edss)}")
print(f"Patients in data/corrected:       {corrected_patient_count}")

---